# AstroLogic — RL Training on Kaggle

Trains **30 hyperparameter experiments** across 3 algorithms:

| Algorithm | Runs | Action space | Budget |
|-----------|------|--------------|--------|
| DQN | 10 | `Discrete(1080)` wrapper | 500 K timesteps |
| PPO | 10 | `MultiDiscrete` native | 500 K timesteps |
| REINFORCE | 10 | `MultiDiscrete` native | 5 000 episodes |

**Environment**: `AstroExploration-v0` — spacecraft navigates the solar system to detect biosignatures on Mars, Europa, and Enceladus.

> Set `FAST_MODE = True` in the config cell for a quick smoke-test (50 K steps / 500 episodes).

In [1]:
# pygame excluded — headless Kaggle kernel
!pip install -q gymnasium stable-baselines3 torch tensorboard tqdm pandas matplotlib seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 4.4 MB/s eta 0:00:0000:01


In [2]:
import os, sys

FAST_MODE = False          # True → 50 K timesteps / 500 episodes for quick test
TOTAL_TIMESTEPS   = 50_000 if FAST_MODE else 500_000
REINFORCE_EPISODES = 500   if FAST_MODE else 5_000
SEED = 42

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

for d in ["astro_env", "agents", "training", "evaluation",
          "results/logs", "results/models", "results/plots", "results/diagrams"]:
    os.makedirs(d, exist_ok=True)

print(f"FAST_MODE={FAST_MODE}  TOTAL_TIMESTEPS={TOTAL_TIMESTEPS}  REINFORCE_EPISODES={REINFORCE_EPISODES}")


FAST_MODE=False  TOTAL_TIMESTEPS=500000  REINFORCE_EPISODES=5000


In [3]:
%%writefile astro_env/__init__.py
"""AstroExploration Gymnasium Environment Package."""

from gymnasium.envs.registration import register

register(
    id="AstroExploration-v0",
    entry_point="astro_env.astro_exploration_env:AstroExplorationEnv",
    max_episode_steps=100_000,
)


Writing astro_env/__init__.py


In [4]:
%%writefile astro_env/celestial_bodies.py
"""Celestial body data for the AstroExploration environment."""

import numpy as np

CELESTIAL_BODIES = {
    "Sun": {"mass": 1.0, "radius": 0.00465, "orbit_radius": 0.0, "orbit_period": 0.0,
            "initial_angle": 0.0, "color": (255, 223, 0), "parent": None,
            "biosignatures": [], "detection_zone_radius": 0.0},
    "Earth": {"mass": 3.003e-6, "radius": 0.0000426, "orbit_radius": 1.0,
             "orbit_period": 365.25, "initial_angle": 0.0, "color": (70, 130, 200),
             "parent": "Sun", "biosignatures": [], "detection_zone_radius": 0.0},
    "Mars": {"mass": 3.213e-7, "radius": 0.0000227, "orbit_radius": 1.524,
            "orbit_period": 687.0, "initial_angle": np.pi / 4, "color": (193, 68, 14),
            "parent": "Sun", "biosignatures": ["ice", "organic_compounds"],
            "detection_zone_radius": 0.05},
    "Jupiter": {"mass": 9.543e-4, "radius": 0.000467, "orbit_radius": 5.203,
               "orbit_period": 4332.59, "initial_angle": np.pi / 3, "color": (201, 176, 131),
               "parent": "Sun", "biosignatures": [], "detection_zone_radius": 0.0},
    "Europa": {"mass": 2.528e-8, "radius": 0.0000104, "orbit_radius": 0.0045,
              "orbit_period": 3.55, "initial_angle": 0.0, "color": (200, 220, 240),
              "parent": "Jupiter",
              "biosignatures": ["liquid_water", "organic_compounds"],
              "detection_zone_radius": 0.03},
    "Saturn": {"mass": 2.857e-4, "radius": 0.000389, "orbit_radius": 9.537,
              "orbit_period": 10759.22, "initial_angle": 2 * np.pi / 3, "color": (210, 180, 100),
              "parent": "Sun", "biosignatures": [], "detection_zone_radius": 0.0},
    "Enceladus": {"mass": 1.08e-10, "radius": 0.00000168, "orbit_radius": 0.00159,
                 "orbit_period": 1.37, "initial_angle": np.pi / 2, "color": (180, 200, 220),
                 "parent": "Saturn",
                 "biosignatures": ["liquid_water", "ice", "signs_of_intelligence"],
                 "detection_zone_radius": 0.02},
}

BIOSIGNATURE_REWARDS = {
    "liquid_water": 500.0, "ice": 300.0,
    "organic_compounds": 750.0, "signs_of_intelligence": 5000.0,
}

TARGET_BODIES = ["Mars", "Europa", "Enceladus"]

INSTRUMENTS = {
    0: {"name": "None", "detects": []},
    1: {"name": "Spectrometer", "detects": ["liquid_water", "organic_compounds"]},
    2: {"name": "ThermalImager", "detects": ["ice", "liquid_water"]},
    3: {"name": "Drill", "detects": ["organic_compounds", "ice", "signs_of_intelligence"]},
}


Writing astro_env/celestial_bodies.py


In [5]:
%%writefile astro_env/physics.py
"""Physics helpers for orbital mechanics simulation."""

import numpy as np

G_NORMALIZED = 4.0 * np.pi**2 / 365.25**2


def gravitational_acceleration(pos, body_pos, body_mass):
    """Compute gravitational acceleration on spacecraft from a single body."""
    direction = body_pos - pos
    distance = np.linalg.norm(direction)
    if distance < 1e-8:
        return np.zeros(3)
    return G_NORMALIZED * body_mass / (distance**2) * (direction / distance)


def orientation_to_direction(pitch, yaw, roll):
    """Convert Euler angles (radians) to a unit forward direction vector."""
    cos_yaw, sin_yaw = np.cos(yaw), np.sin(yaw)
    cos_pitch, sin_pitch = np.cos(pitch), np.sin(pitch)
    forward = np.array([cos_pitch * cos_yaw, cos_pitch * sin_yaw, sin_pitch])
    return forward / (np.linalg.norm(forward) + 1e-8)


def compute_orbital_position(orbit_radius, orbit_period, time, initial_angle, parent_pos=None):
    """Compute position on a circular orbit at a given time."""
    if orbit_period <= 0 or orbit_radius <= 0:
        return parent_pos.copy() if parent_pos is not None else np.zeros(3)
    angle = initial_angle + 2.0 * np.pi * time / orbit_period
    pos = np.array([orbit_radius * np.cos(angle), orbit_radius * np.sin(angle), 0.0])
    if parent_pos is not None:
        pos += parent_pos
    return pos


def normalize_distance(distance, max_distance=50.0):
    """Normalize distance to [0, 1] range where closer = higher value."""
    return max(0.0, 1.0 - distance / max_distance)


def compute_heading(from_pos, to_pos):
    """Compute unit heading vector from one position to another."""
    direction = to_pos - from_pos
    dist = np.linalg.norm(direction)
    if dist < 1e-8:
        return np.zeros(3)
    return direction / dist


Writing astro_env/physics.py


In [6]:
%%writefile astro_env/rewards.py
"""Reward calculator for the AstroExploration environment."""

import numpy as np
from astro_env.celestial_bodies import BIOSIGNATURE_REWARDS


class RewardCalculator:
    """Computes rewards based on environment state transitions."""

    def __init__(self, step_fuel_penalty=0.01, step_time_penalty=0.001,
                 collision_penalty=-1000.0, orbital_insertion_bonus=100.0,
                 transmission_bonus=50.0, proximity_scale=0.1):
        self.step_fuel_penalty = step_fuel_penalty
        self.step_time_penalty = step_time_penalty
        self.collision_penalty = collision_penalty
        self.orbital_insertion_bonus = orbital_insertion_bonus
        self.transmission_bonus = transmission_bonus
        self.proximity_scale = proximity_scale

    def compute(self, state):
        """Compute total reward and breakdown info dict."""
        info = {}
        total = 0.0
        fuel_penalty = -(self.step_fuel_penalty * state.get('fuel_used', 0.0))
        time_penalty = -self.step_time_penalty
        info["reward_step_fuel"] = fuel_penalty
        info["reward_step_time"] = time_penalty
        total += fuel_penalty + time_penalty
        for biosig in state.get('new_biosignatures', []):
            reward = BIOSIGNATURE_REWARDS.get(biosig, 0.0)
            info[f'reward_detect_{biosig}'] = reward
            total += reward
        for biosig in state.get('new_transmissions', []):
            info[f'reward_transmit_{biosig}'] = self.transmission_bonus
            total += self.transmission_bonus
        if state.get('orbital_insertion', False):
            info["reward_orbital_insertion"] = self.orbital_insertion_bonus
            total += self.orbital_insertion_bonus
        if state.get('collision', False) or state.get('out_of_bounds', False):
            info["reward_collision"] = self.collision_penalty
            total += self.collision_penalty
        min_dist = state.get('min_target_distance', 50.0)
        if min_dist < 5.0:
            proximity_reward = self.proximity_scale * (1.0 / (min_dist + 0.1) - 1.0 / 5.1)
            proximity_reward = max(0.0, proximity_reward)
            info["reward_proximity"] = proximity_reward
            total += proximity_reward
        info["reward_total"] = total
        return total, info


Writing astro_env/rewards.py


In [7]:
%%writefile astro_env/wrappers.py
"""Action space wrappers for compatibility with different RL algorithms."""

import numpy as np
import gymnasium as gym
from gymnasium import spaces


class FlattenMultiDiscreteToDiscrete(gym.ActionWrapper):
    """Converts MultiDiscrete action space to Discrete for DQN.

    For MultiDiscrete([5, 3, 3, 3, 4, 2]):
        Total combinations = 5*3*3*3*4*2 = 1080
    """

    def __init__(self, env):
        super().__init__(env)
        assert isinstance(env.action_space, spaces.MultiDiscrete)
        self.nvec = env.action_space.nvec
        self.total_actions = int(np.prod(self.nvec))
        self.action_space = spaces.Discrete(self.total_actions)

    def action(self, flat_action):
        """Decode flat integer to multi-action array."""
        multi_action = np.zeros(len(self.nvec), dtype=np.int64)
        remaining = flat_action
        for i in range(len(self.nvec) - 1, -1, -1):
            multi_action[i] = remaining % self.nvec[i]
            remaining //= self.nvec[i]
        return multi_action

    def reverse_action(self, multi_action):
        """Encode multi-action array to flat integer."""
        flat, multiplier = 0, 1
        for i in range(len(self.nvec) - 1, -1, -1):
            flat += int(multi_action[i]) * multiplier
            multiplier *= self.nvec[i]
        return flat


Writing astro_env/wrappers.py


In [8]:
%%writefile astro_env/astro_exploration_env.py
"""AstroExploration Gymnasium Environment."""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
from astro_env.celestial_bodies import CELESTIAL_BODIES, TARGET_BODIES, INSTRUMENTS
from astro_env.physics import (
    gravitational_acceleration, orientation_to_direction,
    compute_orbital_position, normalize_distance, compute_heading,
)
from astro_env.rewards import RewardCalculator


class AstroExplorationEnv(gym.Env):
    """Custom Gymnasium environment for astrobiological exploration."""

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}
    THRUST_LEVELS = [0.0, 0.25, 0.5, 0.75, 1.0]
    ROTATION_DELTAS = [-5.0, 0.0, 5.0]
    MAX_THRUST = 0.001
    FUEL_CONSUMPTION_RATE = 0.0005
    BATTERY_DRAIN_RATE = 0.00005
    SOLAR_RECHARGE_RANGE = 1.5
    SOLAR_RECHARGE_RATE = 0.0001
    DT = 1.0
    SNR_DETECTION_THRESHOLD = 0.5
    BIOSIG_SUCCESS_COUNT = 3
    MAX_DISTANCE = 50.0

    def __init__(self, render_mode=None, max_episode_steps=100_000):
        super().__init__()
        self.render_mode = render_mode
        self.max_episode_steps = max_episode_steps
        obs_low = np.array(
            [-50, -50, -50, -10, -10, -10, 0, -1, -1, -1, 0, -1, -1, -1,
             0, -1, -1, -1, 0, 0, 0, 0, 0], dtype=np.float32)
        obs_high = np.array(
            [50, 50, 50, 10, 10, 10, 1, 1, 1, 1, 1, 1, 1, 1,
             1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=np.float32)
        self.observation_space = spaces.Box(obs_low, obs_high, dtype=np.float32)
        self.action_space = spaces.MultiDiscrete([5, 3, 3, 3, 4, 2])
        self.reward_calculator = RewardCalculator()
        self._renderer = None
        self.position = self.velocity = self.orientation = None
        self.fuel = self.battery = self.current_step = self.sim_time = None
        self.biosignatures_found = self.biosignatures_transmitted = None
        self.active_instrument = self.body_positions = None
        self.cumulative_reward = self.trajectory = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        earth_data = CELESTIAL_BODIES["Earth"]
        self.position = np.array([earth_data['orbit_radius'], 0.0, 0.0], dtype=np.float64)
        v_earth = 2.0 * np.pi * earth_data['orbit_radius'] / earth_data['orbit_period']
        self.velocity = np.array([0.0, v_earth, 0.0], dtype=np.float64)
        self.orientation = np.array([0.0, 0.0, 0.0], dtype=np.float64)
        self.fuel = 1.0
        self.battery = 1.0
        self.current_step = 0
        self.sim_time = 0.0
        self.biosignatures_found = set()
        self.biosignatures_transmitted = set()
        self.active_instrument = 0
        self.cumulative_reward = 0.0
        self.trajectory = [self.position.copy()]
        if self.np_random is not None:
            for name, body in CELESTIAL_BODIES.items():
                if name != "Sun":
                    body['initial_angle'] += self.np_random.uniform(-0.1, 0.1)
        self._update_body_positions()
        return self._get_obs(), self._get_info()

    def step(self, action):
        assert self.action_space.contains(action)
        thrust_idx, pitch_idx, yaw_idx, roll_idx, instrument_idx, comm_idx = action
        thrust_level = self.THRUST_LEVELS[thrust_idx]
        self.orientation[0] += np.radians(self.ROTATION_DELTAS[pitch_idx])
        self.orientation[1] += np.radians(self.ROTATION_DELTAS[yaw_idx])
        self.orientation[2] += np.radians(self.ROTATION_DELTAS[roll_idx])
        self.active_instrument = instrument_idx
        fuel_used = 0.0
        if thrust_level > 0 and self.fuel > 0:
            forward = orientation_to_direction(*self.orientation)
            self.velocity += forward * thrust_level * self.MAX_THRUST * self.DT
            fuel_used = thrust_level * self.FUEL_CONSUMPTION_RATE
            self.fuel = max(0.0, self.fuel - fuel_used)
        total_grav = np.zeros(3)
        for name, body in CELESTIAL_BODIES.items():
            body_pos = np.zeros(3) if name == "Sun" else self.body_positions[name]
            total_grav += gravitational_acceleration(self.position, body_pos, body['mass'])
        self.velocity += total_grav * self.DT
        self.position += self.velocity * self.DT
        self.sim_time += self.DT
        self._update_body_positions()
        self.battery -= self.BATTERY_DRAIN_RATE
        if np.linalg.norm(self.position) < self.SOLAR_RECHARGE_RANGE:
            self.battery += self.SOLAR_RECHARGE_RATE
        self.battery = np.clip(self.battery, 0.0, 1.0)
        snr = self._compute_snr()
        new_biosigs = []
        if instrument_idx > 0 and snr >= self.SNR_DETECTION_THRESHOLD:
            new_biosigs = self._attempt_detection(instrument_idx)
        new_tx = []
        if comm_idx == 1:
            for b in list(self.biosignatures_found):
                if b not in self.biosignatures_transmitted:
                    self.biosignatures_transmitted.add(b)
                    new_tx.append(b)
        orbital_insertion = self._check_orbital_insertion()
        collision = self._check_collision()
        out_of_bounds = np.linalg.norm(self.position) > self.MAX_DISTANCE
        resource_depleted = self.fuel <= 0 or self.battery <= 0
        success = len(self.biosignatures_transmitted) >= self.BIOSIG_SUCCESS_COUNT
        self.current_step += 1
        truncated = self.current_step >= self.max_episode_steps
        terminated = collision or out_of_bounds or resource_depleted or success
        min_target_dist = self._min_target_distance()
        reward_state = {
            "fuel_used": fuel_used, "collision": collision,
            "out_of_bounds": out_of_bounds, "new_biosignatures": new_biosigs,
            "new_transmissions": new_tx, "orbital_insertion": orbital_insertion,
            "min_target_distance": min_target_dist,
        }
        reward, reward_info = self.reward_calculator.compute(reward_state)
        self.cumulative_reward += reward
        self.trajectory.append(self.position.copy())
        if len(self.trajectory) > 500:
            self.trajectory.pop(0)
        obs = self._get_obs()
        info = self._get_info()
        info.update(reward_info)
        info["success"] = success
        info["collision"] = collision
        info["out_of_bounds"] = out_of_bounds
        info["resource_depleted"] = resource_depleted
        return obs, reward, terminated, truncated, info

    def _update_body_positions(self):
        self.body_positions = {}
        for name, body in CELESTIAL_BODIES.items():
            if name == "Sun":
                self.body_positions[name] = np.zeros(3)
            elif body['parent'] == 'Sun':
                self.body_positions[name] = compute_orbital_position(
                    body['orbit_radius'], body['orbit_period'],
                    self.sim_time, body['initial_angle'])
        for name, body in CELESTIAL_BODIES.items():
            if body['parent'] and body['parent'] != 'Sun':
                parent_pos = self.body_positions.get(body['parent'], np.zeros(3))
                self.body_positions[name] = compute_orbital_position(
                    body['orbit_radius'], body['orbit_period'],
                    self.sim_time, body['initial_angle'], parent_pos=parent_pos)

    def _get_obs(self):
        obs = np.zeros(23, dtype=np.float32)
        obs[0:3] = self.position.astype(np.float32)
        obs[3:6] = np.clip(self.velocity, -10, 10).astype(np.float32)
        for i, target_name in enumerate(TARGET_BODIES):
            target_pos = self.body_positions.get(target_name, np.zeros(3))
            dist = np.linalg.norm(target_pos - self.position)
            heading = compute_heading(self.position, target_pos)
            base_idx = 6 + i * 4
            obs[base_idx] = normalize_distance(dist)
            obs[base_idx + 1:base_idx + 4] = heading.astype(np.float32)
        obs[18] = self.fuel
        obs[19] = self.battery
        obs[20] = self._compute_snr()
        obs[21] = len(self.biosignatures_found) / self.BIOSIG_SUCCESS_COUNT
        obs[22] = len(self.biosignatures_transmitted) / self.BIOSIG_SUCCESS_COUNT
        return obs

    def _get_info(self):
        return {
            "step": self.current_step, "fuel": self.fuel, "battery": self.battery,
            "position": self.position.copy(), "velocity": self.velocity.copy(),
            "biosig_found": list(self.biosignatures_found),
            "biosig_transmitted": list(self.biosignatures_transmitted),
            "cumulative_reward": self.cumulative_reward,
            "active_instrument": INSTRUMENTS[self.active_instrument]["name"],
        }

    def _compute_snr(self):
        max_snr = 0.0
        for target_name in TARGET_BODIES:
            body = CELESTIAL_BODIES[target_name]
            target_pos = self.body_positions.get(target_name, np.zeros(3))
            dist = np.linalg.norm(target_pos - self.position)
            zone_radius = body['detection_zone_radius']
            if zone_radius > 0 and dist < zone_radius:
                max_snr = max(max_snr, 1.0 - (dist / zone_radius))
        return float(max_snr)

    def _attempt_detection(self, instrument_idx):
        detected = []
        instrument = INSTRUMENTS[instrument_idx]
        for target_name in TARGET_BODIES:
            body = CELESTIAL_BODIES[target_name]
            target_pos = self.body_positions.get(target_name, np.zeros(3))
            dist = np.linalg.norm(target_pos - self.position)
            if dist < body['detection_zone_radius']:
                for biosig in body['biosignatures']:
                    if biosig in instrument['detects'] and biosig not in self.biosignatures_found:
                        self.biosignatures_found.add(biosig)
                        detected.append(biosig)
        return detected

    def _check_collision(self):
        for name, body in CELESTIAL_BODIES.items():
            body_pos = self.body_positions.get(name, np.zeros(3))
            dist = np.linalg.norm(body_pos - self.position)
            if dist < max(body['radius'] * 10, 0.001):
                return True
        return False

    def _check_orbital_insertion(self):
        for target_name in TARGET_BODIES:
            target_pos = self.body_positions.get(target_name, np.zeros(3))
            dist = np.linalg.norm(target_pos - self.position)
            if (dist < CELESTIAL_BODIES[target_name]['detection_zone_radius'] * 0.5
                    and np.linalg.norm(self.velocity) < 0.01):
                return True
        return False

    def _min_target_distance(self):
        return min(
            np.linalg.norm(self.body_positions.get(t, np.zeros(3)) - self.position)
            for t in TARGET_BODIES)

    def render(self):
        if self.render_mode is None:
            return None
        if self._renderer is None:
            from visualization.renderer import PygameRenderer
            self._renderer = PygameRenderer(self)
        return self._renderer.render_frame(self._get_render_state())

    def _get_render_state(self):
        return {
            "position": self.position.copy(), "velocity": self.velocity.copy(),
            "orientation": self.orientation.copy(), "fuel": self.fuel,
            "battery": self.battery, "snr": self._compute_snr(),
            "active_instrument": INSTRUMENTS[self.active_instrument]["name"],
            "biosig_found": list(self.biosignatures_found),
            "biosig_transmitted": list(self.biosignatures_transmitted),
            "body_positions": {k: v.copy() for k, v in self.body_positions.items()},
            "trajectory": [p.copy() for p in self.trajectory],
            "current_step": self.current_step, "max_steps": self.max_episode_steps,
            "cumulative_reward": self.cumulative_reward, "thrust_level": 0.0,
        }

    def close(self):
        if self._renderer is not None:
            self._renderer.close()
            self._renderer = None


Writing astro_env/astro_exploration_env.py


In [11]:
%%writefile agents/__init__.py


UsageError: %%writefile is a cell magic, but the cell body is empty.


In [12]:
%%writefile agents/reinforce_policy.py
"""PyTorch policy network for REINFORCE algorithm."""

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical


class REINFORCEPolicy(nn.Module):
    """Multi-head policy for MultiDiscrete action space."""

    def __init__(self, obs_dim=23, action_nvec=None, hidden_sizes=None):
        super().__init__()
        if action_nvec is None:
            action_nvec = [5, 3, 3, 3, 4, 2]
        if hidden_sizes is None:
            hidden_sizes = [128, 64]
        self.action_nvec = action_nvec
        layers, prev_size = [], obs_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_size, h), nn.ReLU()]
            prev_size = h
        self.backbone = nn.Sequential(*layers)
        self.heads = nn.ModuleList([nn.Linear(prev_size, n) for n in action_nvec])

    def forward(self, obs):
        features = self.backbone(obs)
        return [head(features) for head in self.heads]

    def get_action(self, obs):
        logits_list = self.forward(obs)
        actions, total_log_prob = [], torch.tensor(0.0)
        for logits in logits_list:
            dist = Categorical(logits=logits)
            action = dist.sample()
            total_log_prob = total_log_prob + dist.log_prob(action)
            actions.append(action.item())
        return np.array(actions, dtype=np.int64), total_log_prob

    def evaluate_actions(self, obs, actions):
        logits_list = self.forward(obs)
        total_log_prob = torch.zeros(obs.shape[0])
        for i, logits in enumerate(logits_list):
            dist = Categorical(logits=logits)
            total_log_prob = total_log_prob + dist.log_prob(actions[:, i])
        return total_log_prob


Writing agents/reinforce_policy.py


In [13]:
%%writefile agents/reinforce_agent.py
"""REINFORCE (Monte Carlo Policy Gradient) training agent."""

import numpy as np
import torch
import torch.optim as optim
import csv, os
from agents.reinforce_policy import REINFORCEPolicy


class REINFORCEAgent:
    """REINFORCE with optional baseline subtraction."""

    def __init__(self, env, learning_rate=1e-3, gamma=0.99,
                 hidden_sizes=None, baseline='mean', seed=42):
        self.env = env
        self.gamma = gamma
        self.baseline = baseline
        torch.manual_seed(seed)
        np.random.seed(seed)
        self.policy = REINFORCEPolicy(
            obs_dim=env.observation_space.shape[0],
            action_nvec=list(env.action_space.nvec),
            hidden_sizes=hidden_sizes or [128, 64],
        )
        self.optimizer = optim.Adam(self.policy.parameters(), lr=learning_rate)
        self.running_return = 0.0
        self.running_count = 0

    def collect_episode(self):
        obs, info = self.env.reset()
        log_probs, rewards, done = [], [], False
        while not done:
            obs_tensor = torch.FloatTensor(obs)
            action, log_prob = self.policy.get_action(obs_tensor)
            log_probs.append(log_prob)
            obs, reward, terminated, truncated, info = self.env.step(action)
            rewards.append(reward)
            done = terminated or truncated
        return log_probs, rewards, len(rewards), info

    def compute_returns(self, rewards):
        returns, G = [], 0.0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)
        if self.baseline == 'mean':
            if returns.std() > 1e-8:
                returns = (returns - returns.mean()) / (returns.std() + 1e-8)
            else:
                returns = returns - returns.mean()
        elif self.baseline == 'running':
            self.running_count += 1
            self.running_return = 0.95 * self.running_return + 0.05 * returns.mean().item()
            returns = returns - self.running_return
        return returns

    def update(self, log_probs, returns):
        policy_loss = torch.tensor(0.0)
        for lp, G in zip(log_probs, returns):
            policy_loss = policy_loss + (-lp * G)
        policy_loss = policy_loss / len(log_probs)
        self.optimizer.zero_grad()
        policy_loss.backward()
        self.optimizer.step()
        return policy_loss.item()

    def train(self, num_episodes=5000, log_interval=100, save_dir=None):
        history, reward_window = [], []
        for episode in range(1, num_episodes + 1):
            log_probs, rewards, ep_len, info = self.collect_episode()
            ep_reward = sum(rewards)
            returns = self.compute_returns(rewards)
            loss = self.update(log_probs, returns)
            history.append((ep_reward, ep_len))
            reward_window.append(ep_reward)
            if len(reward_window) > 100:
                reward_window.pop(0)
            if episode % log_interval == 0:
                print(f'Episode {episode:5d} | '
                      f'Avg Reward: {np.mean(reward_window):10.2f} | '
                      f'Avg Length: {np.mean([h[1] for h in history[-100:]]):8.1f} | '
                      f'Loss: {loss:10.4f}')
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            torch.save(self.policy.state_dict(), os.path.join(save_dir, 'policy.pt'))
            csv_path = os.path.join(save_dir, 'rewards.csv')
            with open(csv_path, 'w', newline='') as f:
                w = csv.writer(f)
                w.writerow(['episode', 'reward', 'episode_length'])
                for i, (r, l) in enumerate(history, 1):
                    w.writerow([i, r, l])
            print(f'Model saved to {save_dir}/policy.pt')
        return history


Writing agents/reinforce_agent.py


In [14]:
%%writefile training/hyperparams.py
"""Hyperparameter configurations for all 30 experiments."""

import os
# Override timesteps/episodes from notebook-level globals if set
TOTAL_TIMESTEPS    = int(os.environ.get('TOTAL_TIMESTEPS', 500_000))
EVAL_FREQ          = 10_000
N_EVAL_EPISODES    = 5
REINFORCE_EPISODES = int(os.environ.get('REINFORCE_EPISODES', 5_000))

DQN_CONFIGS = [
    {"name": "dqn_baseline",    "learning_rate": 1e-4, "buffer_size": 100_000, "batch_size": 64,  "tau": 1.0,   "gamma": 0.99, "exploration_fraction": 0.3,  "exploration_final_eps": 0.05, "learning_starts": 5000, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "dqn_small_net",   "learning_rate": 1e-4, "buffer_size": 100_000, "batch_size": 64,  "tau": 1.0,   "gamma": 0.99, "exploration_fraction": 0.3,  "exploration_final_eps": 0.05, "learning_starts": 5000, "policy_kwargs": {"net_arch": [128, 128]}},
    {"name": "dqn_high_lr",     "learning_rate": 5e-4, "buffer_size": 100_000, "batch_size": 64,  "tau": 1.0,   "gamma": 0.99, "exploration_fraction": 0.3,  "exploration_final_eps": 0.05, "learning_starts": 5000, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "dqn_low_lr",      "learning_rate": 1e-5, "buffer_size": 100_000, "batch_size": 64,  "tau": 1.0,   "gamma": 0.99, "exploration_fraction": 0.3,  "exploration_final_eps": 0.05, "learning_starts": 5000, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "dqn_large_buffer","learning_rate": 1e-4, "buffer_size": 500_000, "batch_size": 128, "tau": 1.0,   "gamma": 0.99, "exploration_fraction": 0.5,  "exploration_final_eps": 0.02, "learning_starts": 5000, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "dqn_soft_update", "learning_rate": 1e-4, "buffer_size": 100_000, "batch_size": 64,  "tau": 0.005, "gamma": 0.99, "exploration_fraction": 0.3,  "exploration_final_eps": 0.05, "learning_starts": 5000, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "dqn_low_gamma",   "learning_rate": 1e-4, "buffer_size": 100_000, "batch_size": 64,  "tau": 1.0,   "gamma": 0.95, "exploration_fraction": 0.3,  "exploration_final_eps": 0.05, "learning_starts": 5000, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "dqn_deep_net",    "learning_rate": 1e-4, "buffer_size": 100_000, "batch_size": 64,  "tau": 1.0,   "gamma": 0.99, "exploration_fraction": 0.3,  "exploration_final_eps": 0.05, "learning_starts": 5000, "policy_kwargs": {"net_arch": [256, 256, 128]}},
    {"name": "dqn_med_lr_batch","learning_rate": 3e-4, "buffer_size": 200_000, "batch_size": 256, "tau": 1.0,   "gamma": 0.99, "exploration_fraction": 0.3,  "exploration_final_eps": 0.05, "learning_starts": 5000, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "dqn_long_explore","learning_rate": 1e-4, "buffer_size": 100_000, "batch_size": 64,  "tau": 1.0,   "gamma": 0.99, "exploration_fraction": 0.7,  "exploration_final_eps": 0.01, "learning_starts": 5000, "policy_kwargs": {"net_arch": [256, 256]}},
]

PPO_CONFIGS = [
    {"name": "ppo_baseline",     "learning_rate": 3e-4, "n_steps": 2048, "batch_size": 64,  "n_epochs": 10, "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5, "gamma": 0.99, "gae_lambda": 0.95, "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "ppo_high_entropy", "learning_rate": 3e-4, "n_steps": 2048, "batch_size": 64,  "n_epochs": 10, "clip_range": 0.2, "ent_coef": 0.05, "vf_coef": 0.5, "gamma": 0.99, "gae_lambda": 0.95, "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "ppo_low_lr",       "learning_rate": 1e-4, "n_steps": 2048, "batch_size": 64,  "n_epochs": 10, "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5, "gamma": 0.99, "gae_lambda": 0.95, "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "ppo_tight_clip",   "learning_rate": 3e-4, "n_steps": 2048, "batch_size": 64,  "n_epochs": 10, "clip_range": 0.1, "ent_coef": 0.01, "vf_coef": 0.5, "gamma": 0.99, "gae_lambda": 0.95, "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "ppo_wide_clip",    "learning_rate": 3e-4, "n_steps": 2048, "batch_size": 64,  "n_epochs": 10, "clip_range": 0.3, "ent_coef": 0.01, "vf_coef": 0.5, "gamma": 0.99, "gae_lambda": 0.95, "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "ppo_more_epochs",  "learning_rate": 3e-4, "n_steps": 2048, "batch_size": 64,  "n_epochs": 20, "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5, "gamma": 0.99, "gae_lambda": 0.95, "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "ppo_small_net",    "learning_rate": 3e-4, "n_steps": 2048, "batch_size": 64,  "n_epochs": 10, "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5, "gamma": 0.99, "gae_lambda": 0.95, "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [128, 128]}},
    {"name": "ppo_large_rollout","learning_rate": 3e-4, "n_steps": 4096, "batch_size": 128, "n_epochs": 10, "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5, "gamma": 0.99, "gae_lambda": 0.95, "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "ppo_low_gamma",    "learning_rate": 3e-4, "n_steps": 2048, "batch_size": 64,  "n_epochs": 10, "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5, "gamma": 0.95, "gae_lambda": 0.95, "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [256, 256]}},
    {"name": "ppo_high_lr_deep", "learning_rate": 5e-4, "n_steps": 2048, "batch_size": 64,  "n_epochs": 10, "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5, "gamma": 0.99, "gae_lambda": 0.95, "max_grad_norm": 0.5, "policy_kwargs": {"net_arch": [256, 256, 128]}},
]

REINFORCE_CONFIGS = [
    {"name": "reinforce_baseline",          "learning_rate": 1e-3, "gamma": 0.99, "hidden_sizes": [128, 64],      "baseline": "mean"},
    {"name": "reinforce_no_baseline",       "learning_rate": 1e-3, "gamma": 0.99, "hidden_sizes": [128, 64],      "baseline": "none"},
    {"name": "reinforce_low_lr",            "learning_rate": 1e-4, "gamma": 0.99, "hidden_sizes": [128, 64],      "baseline": "mean"},
    {"name": "reinforce_high_lr",           "learning_rate": 5e-3, "gamma": 0.99, "hidden_sizes": [128, 64],      "baseline": "mean"},
    {"name": "reinforce_large_net",         "learning_rate": 1e-3, "gamma": 0.99, "hidden_sizes": [256, 128],     "baseline": "mean"},
    {"name": "reinforce_small_net",         "learning_rate": 1e-3, "gamma": 0.99, "hidden_sizes": [64, 32],       "baseline": "mean"},
    {"name": "reinforce_low_gamma",         "learning_rate": 1e-3, "gamma": 0.95, "hidden_sizes": [128, 64],      "baseline": "mean"},
    {"name": "reinforce_very_low_gamma",    "learning_rate": 1e-3, "gamma": 0.90, "hidden_sizes": [128, 64],      "baseline": "mean"},
    {"name": "reinforce_deep_net",          "learning_rate": 1e-3, "gamma": 0.99, "hidden_sizes": [256, 128, 64], "baseline": "mean"},
    {"name": "reinforce_running_baseline",  "learning_rate": 3e-4, "gamma": 0.99, "hidden_sizes": [128, 64],      "baseline": "running"},
]


Writing training/hyperparams.py


In [15]:
%%writefile training/__init__.py


UsageError: %%writefile is a cell magic, but the cell body is empty.


In [16]:
%%writefile training/train_dqn.py
"""Train DQN agent on AstroExploration environment."""

import os, sys, time, argparse
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import astro_env  # noqa: F401
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from astro_env.wrappers import FlattenMultiDiscreteToDiscrete
from training.hyperparams import DQN_CONFIGS, TOTAL_TIMESTEPS, EVAL_FREQ, N_EVAL_EPISODES


def train_dqn(run_idx, seed=42):
    """Train a single DQN run with the given config index."""
    config = DQN_CONFIGS[run_idx]
    run_name = config['name']
    print(f"\n{'='*60}\nTraining DQN - {run_name} (Run {run_idx})\n{'='*60}")
    log_dir   = f'results/logs/{run_name}'
    model_dir = f'results/models/{run_name}'
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)
    env      = Monitor(FlattenMultiDiscreteToDiscrete(gym.make('AstroExploration-v0')), log_dir)
    eval_env = Monitor(FlattenMultiDiscreteToDiscrete(gym.make('AstroExploration-v0')))
    model_params = {k: v for k, v in config.items() if k != 'name'}
    model = DQN('MlpPolicy', env, verbose=1, seed=seed,
                tensorboard_log=f'results/logs/{run_name}_tb', **model_params)
    eval_callback = EvalCallback(eval_env, best_model_save_path=model_dir,
                                 log_path=log_dir, eval_freq=EVAL_FREQ,
                                 n_eval_episodes=N_EVAL_EPISODES, deterministic=True)
    start = time.time()
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback, progress_bar=True)
    wall_time = time.time() - start
    model.save(os.path.join(model_dir, 'final_model'))
    env.close(); eval_env.close()
    print(f'DQN {run_name} done in {wall_time:.1f}s')
    return {'run_name': run_name, 'wall_time': wall_time, 'algorithm': 'DQN'}


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--run', type=int, default=0)
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args()
    train_dqn(args.run, args.seed)


Writing training/train_dqn.py


In [17]:
%%writefile training/train_ppo.py
"""Train PPO agent on AstroExploration environment."""

import os, sys, time, argparse
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import astro_env  # noqa: F401
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from training.hyperparams import PPO_CONFIGS, TOTAL_TIMESTEPS, EVAL_FREQ, N_EVAL_EPISODES


def train_ppo(run_idx, seed=42):
    """Train a single PPO run with the given config index."""
    config = PPO_CONFIGS[run_idx]
    run_name = config['name']
    print(f"\n{'='*60}\nTraining PPO - {run_name} (Run {run_idx})\n{'='*60}")
    log_dir   = f'results/logs/{run_name}'
    model_dir = f'results/models/{run_name}'
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)
    env      = Monitor(gym.make('AstroExploration-v0'), log_dir)
    eval_env = Monitor(gym.make('AstroExploration-v0'))
    model_params = {k: v for k, v in config.items() if k != 'name'}
    model = PPO('MlpPolicy', env, verbose=1, seed=seed,
                tensorboard_log=f'results/logs/{run_name}_tb', **model_params)
    eval_callback = EvalCallback(eval_env, best_model_save_path=model_dir,
                                 log_path=log_dir, eval_freq=EVAL_FREQ,
                                 n_eval_episodes=N_EVAL_EPISODES, deterministic=True)
    start = time.time()
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback, progress_bar=True)
    wall_time = time.time() - start
    model.save(os.path.join(model_dir, 'final_model'))
    env.close(); eval_env.close()
    print(f'PPO {run_name} done in {wall_time:.1f}s')
    return {'run_name': run_name, 'wall_time': wall_time, 'algorithm': 'PPO'}


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--run', type=int, default=0)
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args()
    train_ppo(args.run, args.seed)


Writing training/train_ppo.py


In [18]:
%%writefile training/train_reinforce.py
"""Train REINFORCE agent on AstroExploration environment."""

import os, sys, time, argparse
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import astro_env  # noqa: F401
import gymnasium as gym
from agents.reinforce_agent import REINFORCEAgent
from training.hyperparams import REINFORCE_CONFIGS, REINFORCE_EPISODES


def train_reinforce(run_idx, seed=42):
    """Train a single REINFORCE run with the given config index."""
    config = REINFORCE_CONFIGS[run_idx]
    run_name = config['name']
    print(f"\n{'='*60}\nTraining REINFORCE - {run_name} (Run {run_idx})\n{'='*60}")
    save_dir = f'results/models/{run_name}'
    os.makedirs(save_dir, exist_ok=True)
    env = gym.make('AstroExploration-v0')
    agent = REINFORCEAgent(
        env=env,
        learning_rate=config['learning_rate'],
        gamma=config['gamma'],
        hidden_sizes=config['hidden_sizes'],
        baseline=config['baseline'],
        seed=seed,
    )
    start = time.time()
    agent.train(num_episodes=REINFORCE_EPISODES, log_interval=100, save_dir=save_dir)
    wall_time = time.time() - start
    env.close()
    print(f'REINFORCE {run_name} done in {wall_time:.1f}s')
    return {'run_name': run_name, 'wall_time': wall_time, 'algorithm': 'REINFORCE'}


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--run', type=int, default=0)
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args()
    train_reinforce(args.run, args.seed)


Writing training/train_reinforce.py


## Verify Environment
Quick smoke-test to confirm the env loads and steps correctly before training.

In [19]:
# Propagate notebook FAST_MODE settings to hyperparams via env vars
import os
os.environ['TOTAL_TIMESTEPS']    = str(TOTAL_TIMESTEPS)
os.environ['REINFORCE_EPISODES'] = str(REINFORCE_EPISODES)

# Register and verify
import importlib, astro_env
importlib.reload(astro_env)
import gymnasium as gym

env = gym.make('AstroExploration-v0')
obs, info = env.reset(seed=0)
print('Observation shape :', obs.shape)
print('Action space      :', env.action_space)
print('Obs space         :', env.observation_space)

for _ in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

env.close()
print('Environment OK — 5 random steps completed.')


Observation shape : (23,)
Action space      : MultiDiscrete([5 3 3 3 4 2])
Obs space         : Box([-50. -50. -50. -10. -10. -10.   0.  -1.  -1.  -1.   0.  -1.  -1.  -1.
   0.  -1.  -1.  -1.   0.   0.   0.   0.   0.], [50. 50. 50. 10. 10. 10.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1.  1.  1.  1.  1.], (23,), float32)
Environment OK — 5 random steps completed.


/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment AstroExploration-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


## Train DQN — 10 configurations

DQN uses a `FlattenMultiDiscreteToDiscrete` wrapper to convert `MultiDiscrete([5,3,3,3,4,2])` → `Discrete(1080)`.

In [ ]:
import sys, os, json, time
sys.path.insert(0, '/kaggle/working')
import astro_env  # registers AstroExploration-v0
from training.train_dqn import train_dqn
from training.hyperparams import DQN_CONFIGS

dqn_results = []
for run_idx in range(len(DQN_CONFIGS)):
    try:
        result = train_dqn(run_idx, seed=SEED)
        result['status'] = 'completed'
    except Exception as e:
        print(f'ERROR DQN run {run_idx}: {e}')
        result = {'run_name': DQN_CONFIGS[run_idx]['name'], 'algorithm': 'DQN',
                  'wall_time': 0, 'status': 'failed', 'error': str(e)}
    dqn_results.append(result)

print(f'\nDQN complete: {sum(r["status"]=="completed" for r in dqn_results)}/'
      f'{len(dqn_results)} runs succeeded')


2026-03-31 09:53:20.133564: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774950800.323792      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774950800.378420      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774950800.862947      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774950800.862988      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774950800.862991      55 computation_placer.cc:177] computation placer alr


Training DQN - dqn_baseline (Run 0)
Using cuda device
Wrapping the env in a DummyVecEnv.
Logging to results/logs/dqn_baseline_tb/DQN_1


Output()

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 526      |
|    ep_rew_mean      | -994     |
|    exploration_rate | 0.987    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1947     |
|    time_elapsed     | 1        |
|    total_timesteps  | 2104     |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 524      |
|    ep_rew_mean      | -995     |
|    exploration_rate | 0.973    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 2045     |
|    time_elapsed     | 2        |
|    total_timesteps  | 4188     |
----------------------------------


...

Eval num_timesteps=240000, episode_reward=-994.88 +/- 0.25

Episode length: 416.20 +/- 1.33

----------------------------------
| eval/               |          |
|    mean_ep_length   | 416      |
|    mean_reward      | -995     |
| rollout/            |          |
|    exploration_rate | 0.05     |
| time/               |          |
|    total_timesteps  | 240000   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0189   |
|    n_updates        | 58749    |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.04e+03 |
|    ep_rew_mean      | -987     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 268      |
|    fps              | 450      |
|    time_elapsed     | 538      |
|    total_timesteps  | 242833   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0424   |
|    n_updates        | 59458    |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.03e+03 |
|    ep_rew_mean      | -987     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 272      |
|    fps              | 452      |
|    time_elapsed     | 545      |
|    total_timesteps  | 246918   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0482   |
|    n_updates        | 60479    |
----------------------------------


## Train PPO — 10 configurations

PPO handles `MultiDiscrete` natively — no wrapper needed.

In [ ]:
from training.train_ppo import train_ppo
from training.hyperparams import PPO_CONFIGS

ppo_results = []
for run_idx in range(len(PPO_CONFIGS)):
    try:
        result = train_ppo(run_idx, seed=SEED)
        result['status'] = 'completed'
    except Exception as e:
        print(f'ERROR PPO run {run_idx}: {e}')
        result = {'run_name': PPO_CONFIGS[run_idx]['name'], 'algorithm': 'PPO',
                  'wall_time': 0, 'status': 'failed', 'error': str(e)}
    ppo_results.append(result)

print(f'\nPPO complete: {sum(r["status"]=="completed" for r in ppo_results)}/'
      f'{len(ppo_results)} runs succeeded')


## Train REINFORCE — 10 configurations

Custom PyTorch implementation — Monte Carlo Policy Gradient with optional baseline subtraction.

In [ ]:
from training.train_reinforce import train_reinforce
from training.hyperparams import REINFORCE_CONFIGS

reinforce_results = []
for run_idx in range(len(REINFORCE_CONFIGS)):
    try:
        result = train_reinforce(run_idx, seed=SEED)
        result['status'] = 'completed'
    except Exception as e:
        print(f'ERROR REINFORCE run {run_idx}: {e}')
        result = {'run_name': REINFORCE_CONFIGS[run_idx]['name'], 'algorithm': 'REINFORCE',
                  'wall_time': 0, 'status': 'failed', 'error': str(e)}
    reinforce_results.append(result)

print(f'\nREINFORCE complete: {sum(r["status"]=="completed" for r in reinforce_results)}/'
      f'{len(reinforce_results)} runs succeeded')


## Save Experiment Index


In [ ]:
all_results = dqn_results + ppo_results + reinforce_results
index = {
    'total_experiments': len(all_results),
    'total_wall_time_seconds': sum(r.get('wall_time', 0) for r in all_results),
    'seed': SEED,
    'fast_mode': FAST_MODE,
    'results': all_results,
}
index_path = 'results/experiment_index.json'
with open(index_path, 'w') as f:
    json.dump(index, f, indent=2)

print('Experiment index saved to', index_path)
print(f'Total wall time: {index["total_wall_time_seconds"]/3600:.2f} h')
completed = [r for r in all_results if r.get('status') == 'completed']
print(f'{len(completed)}/{len(all_results)} experiments completed successfully')


## Results Summary


In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Wall-time bar chart ─────────────────────────────────────────────────────
rows = []
for r in all_results:
    rows.append({'Algorithm': r.get('algorithm', ''),
                 'Run': r.get('run_name', ''),
                 'Wall Time (s)': r.get('wall_time', 0),
                 'Status': r.get('status', '')})
df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Group by algorithm
summary = df[df['Status'] == 'completed'].groupby('Algorithm')['Wall Time (s)'].agg(['mean', 'sum'])
print('\nWall time summary (completed runs):')
print(summary)

# ── Learning curves (per algorithm) ─────────────────────────────────────────
import glob

COLORS = {'DQN': '#e74c3c', 'PPO': '#2ecc71', 'REINFORCE': '#9b59b6'}

def load_monitor(log_dir):
    for pattern in [f'{log_dir}/monitor.csv', f'{log_dir}/*.monitor.csv']:
        files = glob.glob(pattern)
        if files:
            try:
                d = pd.read_csv(files[0], skiprows=1)
                if 'r' in d.columns:
                    d = d.rename(columns={'r': 'reward', 'l': 'length', 't': 'time'})
                return d
            except Exception:
                pass
    return None

def load_reinforce_csv(model_dir):
    path = f'{model_dir}/rewards.csv'
    return pd.read_csv(path) if os.path.exists(path) else None

def smooth(values, w=50):
    return pd.Series(values).rolling(window=min(w, max(1, len(values)//5)), min_periods=1).mean().values

from training.hyperparams import DQN_CONFIGS, PPO_CONFIGS, REINFORCE_CONFIGS

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('AstroLogic — Learning Curves', fontsize=15, fontweight='bold')

algo_data = [
    ('DQN',       DQN_CONFIGS,       axes[0], False),
    ('PPO',       PPO_CONFIGS,       axes[1], False),
    ('REINFORCE', REINFORCE_CONFIGS, axes[2], True),
]

for algo_name, configs, ax, is_reinforce in algo_data:
    ax.set_title(algo_name, fontsize=13)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Episode Reward')
    ax.grid(True, alpha=0.3)
    colors = plt.cm.tab10(range(len(configs)))
    for i, cfg in enumerate(configs):
        run_name = cfg['name']
        if is_reinforce:
            df_run = load_reinforce_csv(f'results/models/{run_name}')
            col = 'reward'
        else:
            df_run = load_monitor(f'results/logs/{run_name}')
            col = 'reward'
        if df_run is not None and col in df_run.columns:
            ax.plot(smooth(df_run[col].values), color=colors[i], alpha=0.75,
                    label=run_name.replace(f'{algo_name.lower()}_', ''), linewidth=0.9)
    ax.legend(fontsize=6, loc='lower right')

plt.tight_layout()
plt.savefig('results/plots/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Learning curves saved.')


## Package Outputs

Zip everything in `results/` so it appears in the Kaggle output panel for download.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/astro_results', 'zip', '/kaggle/working/results')
print('Archive created: /kaggle/working/astro_results.zip')
# List saved models
for root, dirs, files in os.walk('results/models'):
    for f in files:
        fpath = os.path.join(root, f)
        print(f'  {fpath}  ({os.path.getsize(fpath)//1024} KB)')
